# 记忆

[记忆](https://docs.langchain.com/oss/python/langgraph/add-memory)（Memory）是一个可选模块。如非必要，你无需向智能体添加 Memory 模块。因为 StateGraph 本身就含有历史消息列表 `messages`，足以满足最基础的“记忆”需求。

需要添加 Memory 模块的情况包括：

1. 历史消息太多，需要用外部工具存储记忆
2. 触发人工干预（[interrupt](https://docs.langchain.com/oss/python/langgraph/interrupts)），需要临时保存 Agent 的状态
3. 跨对话提取用户偏好 等等

LangGraph 将记忆分为：

- [短期记忆](https://docs.langchain.com/oss/python/langchain/short-term-memory)（MemorySaver）
- [长期记忆](https://docs.langchain.com/oss/python/langchain/long-term-memory)（MemoryStore）

此外，还有一个 [LangMem](https://langchain-ai.github.io/langmem/) 也提供记忆存取功能。

In [15]:
import os
import sqlite3

from dotenv import load_dotenv
from dataclasses import dataclass
from typing_extensions import TypedDict
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

# 加载模型配置
_ = load_dotenv()

# 加载模型
model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.7,
)

# 创建助手节点
def assistant(state: MessagesState):
    return {'messages': [model.invoke(state['messages'])]}

## 一、短期记忆

短期记忆（工作记忆）一般用于临时存储 Agent 或 工作流 的状态，以便在失败或重试后恢复。

### 1）在工作流中使用短期记忆

如果为工作流配置了检查点，下次调用该工作流时，会接着上一次对话内容继续聊下去。如果没有配置，将不会保留历史对话。

In [2]:
# 创建短期记忆
checkpointer = InMemorySaver()

# 创建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node('assistant', assistant)

# 添加边
builder.add_edge(START, 'assistant') # START 节点会自动接收用户输入
builder.add_edge('assistant', END)

# 使用检查点
graph = builder.compile(checkpointer=checkpointer)

# 如果不使用检查点，看看会发生什么？
# graph = builder.compile()

# 告诉智能体我是谁
"""
graph.invoke (...) 运行流程
启动 → 进入 assistant 节点
assistant 节点执行：接收输入，生成 AI 回复
关键点：assistant 节点不会只返回 AI 回复，而是会：
读取当前所有 messages
把新的 AI 消息「追加」到列表末尾
然后把更新后的完整消息列表返回
"""
result = graph.invoke(
    {'messages': ['你好！我是派大星']},
    {"configurable": {"thread_id": "1"}},
)

"""
thread_id: "1" 的作用：给对话起个名字,是LangGraph 实现记忆（多轮对话）最核心的钥匙！
它就像是给你的聊天开了一个房间号：
thread_id = "1" = 1 号房间
thread_id = "2" = 2 号房间
核心功能只有 2 个，必须记住：
1. 存储记忆
你设置了 thread_id: "1"，LangGraph 就会把这一轮的所有对话历史，存在 1 号房间 里。
2. 读取记忆
下一次你再调用 graph.invoke，并且还传 thread_id: "1"，AI 就会去 1 号房间 把之前的聊天记录全部调出来，知道你之前说过什么！
"""

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

你好！我是派大星
================================== Ai Message ==================================

哇！派大星，真的是你吗？！*兴奋地蹦跳* 我超喜欢你的！你是从比基尼海滩游过来找我的吗？要不要一起去海底捞几个水母玩？或者...嗯...你觉得现在去蟹老板那里吃个蟹黄堡怎么样？我请客哦！*期待地看着你*


In [3]:
# 让智能体说出我的名字
result = graph.invoke(
    {"messages": [{"role": "user", "content": "请问我是谁？"}]},
    {"configurable": {"thread_id": "1"}},  
)

for message in result['messages']:
    message.pretty_print()


================================ Human Message =================================

你好！我是派大星
================================== Ai Message ==================================

哇！派大星，真的是你吗？！*兴奋地蹦跳* 我超喜欢你的！你是从比基尼海滩游过来找我的吗？要不要一起去海底捞几个水母玩？或者...嗯...你觉得现在去蟹老板那里吃个蟹黄堡怎么样？我请客哦！*期待地看着你*
================================ Human Message =================================

请问我是谁？
================================== Ai Message ==================================

*揉揉眼睛仔细打量* 哎呀，你不是派大星嘛！我们是最好的朋友啊！住在比基尼海滩的粉红色石头下面的就是你呀！你最爱躺在门前晒太阳，还喜欢和我一起去找水母玩。虽然有时候你不太聪明的样子，但你可是我最棒的朋友！*拍拍你的背* 诶，要不要一起去吃冰淇淋？我记得新开了一家冰激淋店，听说他们的海藻味冰淇淋特别好吃！


### 2）在智能体中使用短期记忆

在智能体中使用短期记忆的效果，和工作流中类似。

In [4]:
from langchain.agents import create_agent

# 创建短期记忆
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    checkpointer=checkpointer
)

# 告诉智能体我是章鱼哥
result = agent.invoke(
    {'messages': ['哈喽！我是章鱼哥']},
    {"configurable": {"thread_id": "2"}},
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

哈喽！我是章鱼哥
================================== Ai Message ==================================

*正在整理领带* 哼，又是一个忙碌的早晨。哦？海之丘便利店今天居然来了新员工？*略带傲慢地瞥了一眼* 我是这里的收银员章鱼哥，如果你需要帮助的话...勉强可以指导你一下，毕竟我可是这里最专业的员工。

*触手轻轻敲打着收银台* 不过别指望我会太热情，我还要准备蟹老板的珍珠奶茶订单呢。说起来你知道怎么分辨新鲜的海藻吗？这可是个技术活。


In [5]:
# 让智能体说出我的名字
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我是谁？"}]},
    {"configurable": {"thread_id": "2"}},  
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

哈喽！我是章鱼哥
================================== Ai Message ==================================

*正在整理领带* 哼，又是一个忙碌的早晨。哦？海之丘便利店今天居然来了新员工？*略带傲慢地瞥了一眼* 我是这里的收银员章鱼哥，如果你需要帮助的话...勉强可以指导你一下，毕竟我可是这里最专业的员工。

*触手轻轻敲打着收银台* 不过别指望我会太热情，我还要准备蟹老板的珍珠奶茶订单呢。说起来你知道怎么分辨新鲜的海藻吗？这可是个技术活。
================================ Human Message =================================

我是谁？
================================== Ai Message ==================================

*推了推眼镜，仔细打量着你* 

哼，又一个不记得自己是谁的家伙？看看你的样子...嗯，穿着这么奇怪的衣服，八成是从比奇堡来的游客吧？

*用触手指了指便利店的方向*

这里是海之丘便利店，如果你真的不记得自己是谁，不如先来杯提神的饮料？刚好今天的海藻拿铁特价，不过要加珍珠的话得另收费...蟹老板定的价格，我也很无奈啊。

*叹了口气*

要不这样，你先在这儿帮忙整理下货架，说不定能想起点什么。当然，工钱可不会太高，毕竟...你也看到了，这年头生意不好做啊。


为了验证 InMemorySaver 是否真的有效，可以将 checkpointer 注释后，再观察智能体的行为。

### 3）使用数据库保存短期记忆

如果用 SQLite 保存工作状态，即使退出程序，应该也能恢复退出之前的状态。下面我们来验证这一点。在此之前，需要安装一个 Python 包以支持 SqliteSaver 检查点：

```bash
pip install langgraph-checkpoint-sqlite
```

In [6]:
# 删除SQLite数据库
if os.path.exists("short-memory.db"):
    os.remove("short-memory.db")

In [7]:
from langgraph.checkpoint.sqlite import SqliteSaver

# 创建sqlite支持的短期记忆
checkpointer = SqliteSaver(
    sqlite3.connect("short-memory.db", check_same_thread=False)
)
"""
check_same_thread=False 到底干嘛的？
它是 SQLite 数据库的安全开关，作用只有一个：
允许 “不同线程” 访问同一个数据库连接
为什么必须加这个？
SQLite 默认规则：
一个数据库连接，只能被创建它的那个线程使用，别的线程碰都不让碰。
LangGraph 规则：
它内部会用多线程运行智能体、保存记忆、执行图流程。
如果不加 check_same_thread=False，LangGraph 别的线程想读写数据库 → 直接报错！
"""

# 创建Agent
agent = create_agent(
    model=model,
    checkpointer=checkpointer,
)

# 告诉智能体我是沙悟净
result = agent.invoke(
    {'messages': ['嗨！我是沙悟净']},
    {"configurable": {"thread_id": "3"}},
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

嗨！我是沙悟净
================================== Ai Message ==================================

*正在河边巡逻的沙僧听到声音，转过身来，手中的降妖杖微微放下*

咦？这位施主面生得很啊，不知可是从东土大唐来的？老沙我在这流沙河畔守候多时，等候取经人。若施主也是往西天求取真经的，不如与我说说来意？

*抬头看了看天空，压低声音*

这河里最近不太平，常有妖怪出没，施主若是单身一人，老沙倒是可以护送一程。


创建一个新的 Agent，并为它配置 SQLite 检查点。看看智能体能否从 SQLite 中读取关于我名字的记忆。

In [8]:
# 创建一个新的Agent
new_agent = create_agent(
    model=model,
    checkpointer=checkpointer,
)

# 让智能体回忆我的名字
result = new_agent.invoke(
    {'messages': ['我是谁？']},
    {"configurable": {"thread_id": "3"}},
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

嗨！我是沙悟净
================================== Ai Message ==================================

*正在河边巡逻的沙僧听到声音，转过身来，手中的降妖杖微微放下*

咦？这位施主面生得很啊，不知可是从东土大唐来的？老沙我在这流沙河畔守候多时，等候取经人。若施主也是往西天求取真经的，不如与我说说来意？

*抬头看了看天空，压低声音*

这河里最近不太平，常有妖怪出没，施主若是单身一人，老沙倒是可以护送一程。
================================ Human Message =================================

我是谁？
================================== Ai Message ==================================

*沙僧眯着眼睛打量着你，手中的降妖杖轻轻颤动*

阿弥陀佛...老沙我虽然记性不太好，但这位施主倒是让我想起一些往事。你可知道，当年我也是因为犯了天条，被贬到这流沙河受苦。如今既已皈依佛门，倒也不敢轻易认人。

*河水忽然泛起涟漪，隐约传来怪声*

嗯？这水声不对劲...施主，你身上似乎有种熟悉的气息，让这河里的妖气都躁动起来了。莫非...你也和那场大劫有关？

*警惕地环顾四周*

若施主当真是取经人，可要小心了。这流沙河可不是好过的地方，老沙我在这里吃了不少人参果，也送了不少人上路。不过既然菩萨指点我在此等候，想必施主的到来必有缘由。


## 二、长期记忆

长期记忆一般用于保存与业务相关的重要信息，比如用户属性、流量参数等。

### 1）创建 Embedding 生成函数

长期记忆支持使用 Embedding 检索语义相近的内容。下面创建一个 Embedding 生成函数，该函数可生成检索所需的 Embedding。

In [9]:
# 嵌入维度
EMBED_DIM = 1024

# 获取text embedding的接口
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# embedding生成函数
def embed(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(
        model="text-embedding-v4",
        input=texts,
        dimensions=EMBED_DIM,
    )
    return [item.embedding for item in response.data]

# 测试能否正常生成text embedding
texts = [
    "LangGraph的中间件非常强大",
    "LangGraph的MCP也很好用",
]
vectors = embed(texts)

len(vectors), len(vectors[0])

(2, 1024)

### 2）读写长期记忆

先向 InMemoryStore 中写入两条数据。

In [10]:
# 创建 InMemoryStore 内存存储
store = InMemoryStore(index={"embed": embed, "dims": EMBED_DIM}) # 创建带向量检索的存储器

# 添加两条用户数据 user_1 user_2
namespace = ("users", )

# 存入用户 1 的偏好规则
store.put(
    namespace,  # Namespace to group related data together
    "user_1",  # Key within the namespace
    {
        "rules": [
            "User likes short, direct language",
            "User only speaks English & python",
        ],
        "rule_id": "3",
    },
)

# 存入用户 2 的基本信息
store.put(
    ("users",),
    "user_2",
    {
        "name": "John Smith",
        "language": "English",
    }
)

通过 namespace 和 key，可以直接读取长期记忆。

In [11]:
item = store.get(namespace, "user_2")
item

Item(namespace=['users'], key='user_2', value={'name': 'John Smith', 'language': 'English'}, created_at='2026-05-02T10:08:53.733656+00:00', updated_at='2026-05-02T10:08:53.733670+00:00')

也可以通过向量检索召回。

In [12]:
items = store.search( 
    namespace,
    query="language preferences",
    filter={"rule_id": "3"},
)
items

[Item(namespace=['users'], key='user_1', value={'rules': ['User likes short, direct language', 'User only speaks English & python'], 'rule_id': '3'}, created_at='2026-05-02T10:08:53.556637+00:00', updated_at='2026-05-02T10:08:53.556643+00:00', score=0.4085710154661828)]

### 3）使用工具读取长期记忆

In [13]:
"""
store 来自 create_agent
context 来自 invoke
runtime 是 LangGraph 自动打包给你的
"""

@dataclass
class Context:
    user_id: str

# 定义工具时，写了 runtime,这句话对 LangGraph 来说等于：“我需要运行时环境，请自动给我注入！”
@tool
def get_user_info(runtime: ToolRuntime[Context]) -> str:
    """用于查询用户信息"""
    user_id = runtime.context.user_id
    user_info = runtime.store.get(("users",), user_id) 
    return str(user_info.value) if user_info else "未知用户"

# 创建Agent
agent = create_agent(
    model=model,
    tools=[get_user_info],# 自动调用工具，把 runtime 塞进去
    store=store, 
    context_schema=Context # 创建 Agent 时，把 store 交给了它
)

# 运行Agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "查阅用户信息"}]},
    context=Context(user_id="user_2")  # 运行时，把 context 交给了 Agent
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

查阅用户信息
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_76262c1a498b4b00893e5ac7)
 Call ID: call_76262c1a498b4b00893e5ac7
  Args:
================================= Tool Message =================================
Name: get_user_info

{'name': 'John Smith', 'language': 'English'}
================================== Ai Message ==================================

用户信息如下：

- **姓名**: John Smith
- **语言**: English


### 4）使用工具写入长期记忆

In [14]:
class UserInfo(TypedDict):
    name: str

@tool
def save_user_info(user_info: UserInfo, runtime: ToolRuntime[Context]) -> str:
    """用于保存/更新用户信息"""
    user_id = runtime.context.user_id
    runtime.store.put(("users",), user_id, user_info) 
    return "成功保存用户信息"

# 创建gent
agent = create_agent(
    model=model,
    tools=[save_user_info],
    store=store,
    context_schema=Context
)

# 运行Agent
agent.invoke(
    {"messages": [{"role": "user", "content": "My name is John Smith"}]},
    context=Context(user_id="user_123") 
)

store.get(("users",), "user_123").value

{'name': 'John Smith'}